[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/main/notebooks/12_Constraints_and_MPC.ipynb)

# DiveLab

## Notebook 12 — Constraints and Model Predictive Control

**Guiding question:** How can the diver choose corrective actions while anticipating future motion and respecting physical limits?

In DiveLab, the **controller is the diver**.

The equations in this notebook do not represent an autonomous diving robot.

They represent a mathematical model of a skilled diver who:

- observes the current situation;
- predicts what may happen next;
- considers physical constraints;
- chooses a correction;
- observes the result;
- repeats the process.

This predictive decision structure is closely related to **Model Predictive Control (MPC)**.

## Learning objectives

By the end of this lab, you will be able to:

- distinguish feedback correction from predictive decision-making;
- define physical and operational constraints;
- understand the basic idea of Model Predictive Control;
- predict future depth and velocity over a finite horizon;
- evaluate candidate control actions using a cost function;
- respect ascent-rate and actuator constraints;
- understand receding-horizon control;
- interpret MPC as a model of constrained diver decision-making.

# 1. The diver as controller

Throughout DiveLab, the controller has always represented the diver.

The diver closes the loop:

```text
environment
    |
    v
physical diver
    |
    v
sensation / instruments
    |
    v
human estimation
    |
    v
decision
    |
    v
BCD / breathing / posture / finning
    |
    +----------------------> physical diver
```

Mathematical controllers such as PD, LQR or MPC are **models of control logic**.

They help us study what good corrective behavior should look like.

# 2. Why feedback alone may not be enough

A simple feedback controller reacts to current error:

$$
u(t)=\phi(e(t)).
$$

For example:

> "I am rising too fast, so I reduce buoyancy."

But a skilled diver also anticipates:

> "If I continue rising at this rate, gas will expand, buoyancy will increase, and the ascent may accelerate."

That is predictive reasoning.

MPC formalizes this idea.

# 3. What is Model Predictive Control?

In simple words:

> **MPC predicts several possible futures, chooses the best feasible action, applies only the first action, then repeats the calculation with new information.**

At each decision step:

1. measure or estimate the current state;
2. predict future motion;
3. consider constraints;
4. evaluate possible control actions;
5. choose the best action;
6. apply the first action;
7. observe the new state;
8. repeat.

This is called **receding-horizon control**.

## Why "receding horizon"?

Suppose the diver predicts the next 10 seconds.

After one second, the diver does not blindly follow the old 10-second plan.

Instead:

- new measurements arrive;
- the state estimate changes;
- disturbances may have occurred;
- the prediction is recomputed.

So the horizon moves forward with time.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 4. Reuse the nonlinear diver model

We use the nonlinear plant developed previously:

$$
\dot z=-v
$$

$$
m\dot v
=
F_B(z,V_s)-mg-F_D(v)+d(t)
$$

$$
\dot V_s=u.
$$

In [ ]:
rho = 1025.0
g = 9.80665
P0 = 101325.0

mass = 90.0
Cd = 0.9
A_drag = 0.7

z_e = 20.0
Vs_e = 0.005

def pressure_at_depth(z):
    return P0 + rho * g * z

def gas_volume_at_depth(z, surface_volume):
    return surface_volume * P0 / pressure_at_depth(z)

Vg_e = gas_volume_at_depth(z_e, Vs_e)
fixed_volume = mass / rho - Vg_e

def buoyant_force(z, Vs):
    Vg = gas_volume_at_depth(z, Vs)
    return rho * g * (fixed_volume + Vg)

def drag_force(v):
    return 0.5 * rho * Cd * A_drag * v * abs(v)

def acceleration(z, v, Vs, disturbance_force=0.0):
    return (
        buoyant_force(z, Vs)
        - mass * g
        - drag_force(v)
        + disturbance_force
    ) / mass

# 5. Constraints

A real diver cannot choose arbitrary states or actions.

We introduce simplified constraints.

### Actuator constraint

The rate of gas addition or venting is limited:

$$
|u|\le u_{\max}.
$$

### Vertical-velocity constraint

We impose a modeling limit:

$$
|v|\le v_{\max}.
$$

### Gas-state constraint

The BCD gas state must remain physically meaningful:

$$
V_s\ge0.
$$

These are **modeling constraints for the control exercise**, not diving-procedure prescriptions.

In [ ]:
u_max = 0.00035
v_max = 0.20

Vs_min = 0.0
Vs_max = 0.012

# 6. Prediction model

MPC needs a model for predicting future motion.

We use the nonlinear equations themselves.

For one timestep $\Delta t$:

$$
v_{k+1}
=
v_k+a_k\Delta t
$$

$$
z_{k+1}
=
z_k-v_{k+1}\Delta t
$$

$$
V_{s,k+1}
=
V_{s,k}+u_k\Delta t.
$$

In [ ]:
def predict_one_step(state, u, dt):
    z, v, Vs = state

    u = np.clip(u, -u_max, u_max)

    a = acceleration(z, v, Vs)

    v_next = v + a * dt
    z_next = max(z - v_next * dt, 0.0)
    Vs_next = np.clip(Vs + u * dt, Vs_min, Vs_max)

    return np.array([z_next, v_next, Vs_next])

# 7. A tracking goal

Suppose the diver wants to follow a gradual ascent reference.

We reuse a simple constant-rate reference:

$$
z_r(t)
$$

with desired upward velocity:

$$
v_r(t).
$$

In [ ]:
z_start = 30.0
z_final = 10.0
v_reference = 0.10

def reference(t):
    z_r = z_start - v_reference * t

    if z_r > z_final:
        return z_r, v_reference

    return z_final, 0.0

# 8. What makes one predicted future better than another?

MPC uses a **cost function**.

A simple cost can penalize:

- depth tracking error;
- velocity tracking error;
- excessive control effort.

For one step:

$$
\ell
=
q_z(z-z_r)^2
+
q_v(v-v_r)^2
+
r_u u^2.
$$

In [ ]:
q_z = 8.0
q_v = 4.0
r_u = 2e6

## Why penalize control effort?

Without a control penalty, the optimizer may constantly choose extreme BCD actions.

The term:

$$
r_u u^2
$$

encourages smoother, smaller corrections.

This is similar to the tradeoff introduced with LQR.

# 9. Constraint penalties

A candidate future that violates an important constraint should be considered undesirable.

For example, if:

$$
|v|>v_{\max},
$$

we add a large penalty.

This is a simple educational alternative to a full constrained optimization solver.

In [ ]:
constraint_penalty = 1e6

def stage_cost(state, u, t):
    z, v, Vs = state
    z_r, v_r = reference(t)

    cost = (
        q_z * (z - z_r)**2
        + q_v * (v - v_r)**2
        + r_u * u**2
    )

    if abs(v) > v_max:
        cost += constraint_penalty * (abs(v) - v_max)**2

    if Vs <= Vs_min or Vs >= Vs_max:
        cost += constraint_penalty

    return cost

# 10. Predict a future trajectory

Given:

- the current state;
- a sequence of candidate control actions;

we can predict the future.

In [ ]:
def rollout(state0, control_sequence, t0, dt):
    states = [np.array(state0, dtype=float)]
    total_cost = 0.0

    state = np.array(state0, dtype=float)

    for j, u in enumerate(control_sequence):
        t_j = t0 + j * dt
        total_cost += stage_cost(state, u, t_j)

        state = predict_one_step(state, u, dt)
        states.append(state)

    return np.array(states), total_cost

# 11. First predictive experiment

Start at:

$$
z=30\ \text{m},
\qquad
v=0.
$$

Compare three possible constant actions over a short prediction horizon:

- vent gas;
- do nothing;
- add gas.

In [ ]:
dt_mpc = 0.25
horizon_steps = 12

state0 = np.array([30.0, 0.0, Vs_e])

candidate_actions = [
    -u_max,
    0.0,
    +u_max
]

for action in candidate_actions:
    seq = np.full(horizon_steps, action)
    states_pred, cost = rollout(state0, seq, 0.0, dt_mpc)

    print(f"u = {action:+.6f} -> predicted cost = {cost:.2f}")

The predictive controller does not ask only:

> "What action reduces the current error?"

It asks:

> "What action produces the best predicted future while respecting constraints?"

# 12. Plot candidate futures

In [ ]:
for action in candidate_actions:
    seq = np.full(horizon_steps, action)
    states_pred, cost = rollout(state0, seq, 0.0, dt_mpc)

    future_t = np.arange(horizon_steps + 1) * dt_mpc
    plt.plot(
        future_t,
        states_pred[:, 0],
        label=f"u={action:+.6f}"
    )

plt.xlabel("Prediction time [s]")
plt.ylabel("Predicted depth [m]")
plt.title("Candidate predicted futures")
plt.grid(True)
plt.legend()
plt.show()

# 13. A simple educational MPC

A full MPC solver optimizes an entire future control sequence.

For this notebook we start with a simpler idea:

- choose from a grid of candidate actions;
- assume that action remains constant over the short horizon;
- predict each future;
- choose the lowest-cost candidate;
- apply that action for only one timestep.

Then repeat.

This keeps the algorithm transparent.

In [ ]:
control_grid = np.linspace(-u_max, u_max, 15)

def simple_mpc_action(state, t0, dt=dt_mpc, horizon=horizon_steps):
    best_u = None
    best_cost = np.inf
    best_states = None

    for u_candidate in control_grid:
        sequence = np.full(horizon, u_candidate)

        states, cost = rollout(
            state,
            sequence,
            t0,
            dt
        )

        if cost < best_cost:
            best_cost = cost
            best_u = u_candidate
            best_states = states

    return best_u, best_cost, best_states

# 14. One MPC decision

In [ ]:
u_choice, cost_choice, prediction_choice = simple_mpc_action(
    state0,
    t0=0.0
)

print(f"Chosen action: {u_choice:+.6f} m^3/s")
print(f"Predicted cost: {cost_choice:.2f}")

The action is chosen by looking ahead.

But only the **first action** is actually applied.

After that, the state is measured again and a new prediction is made.

That is receding-horizon control.

# 15. Closed-loop MPC simulation

In [ ]:
def simulate_mpc(
    z0=30.0,
    v0=0.0,
    Vs0=Vs_e,
    duration=220.0,
    dt=dt_mpc,
):
    n = int(duration / dt) + 1
    t = np.linspace(0, duration, n)

    state_hist = np.zeros((n, 3))
    u_hist = np.zeros(n)
    cost_hist = np.zeros(n)

    state_hist[0] = np.array([z0, v0, Vs0])

    for k in range(n - 1):
        current_state = state_hist[k]

        u, predicted_cost, _ = simple_mpc_action(
            current_state,
            t[k],
            dt=dt,
            horizon=horizon_steps
        )

        state_hist[k + 1] = predict_one_step(
            current_state,
            u,
            dt
        )

        u_hist[k] = u
        cost_hist[k] = predicted_cost

    u_hist[-1] = u_hist[-2]
    cost_hist[-1] = cost_hist[-2]

    return t, state_hist, u_hist, cost_hist

In [ ]:
t_mpc, states_mpc, u_mpc, cost_mpc = simulate_mpc()

z_mpc = states_mpc[:, 0]
v_mpc = states_mpc[:, 1]
Vs_mpc = states_mpc[:, 2]

z_ref_mpc = np.array([reference(tt)[0] for tt in t_mpc])
v_ref_mpc = np.array([reference(tt)[1] for tt in t_mpc])

# 16. Depth tracking with predictive control

In [ ]:
plt.plot(t_mpc, z_ref_mpc, label="Reference")
plt.plot(t_mpc, z_mpc, label="MPC trajectory")

plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Predictive trajectory tracking")
plt.grid(True)
plt.legend()
plt.show()

The controller repeatedly predicts and corrects.

It is not following one fixed precomputed plan.

It continually replans from the current state.

# 17. Velocity and constraint

In [ ]:
plt.plot(t_mpc, v_mpc, label="Actual velocity")
plt.plot(t_mpc, v_ref_mpc, label="Reference velocity")
plt.axhline(v_max, linestyle="--", label="Velocity limit")
plt.axhline(-v_max, linestyle="--")

plt.xlabel("Time [s]")
plt.ylabel("Upward velocity [m/s]")
plt.title("Velocity tracking and constraint")
plt.grid(True)
plt.legend()
plt.show()

In MPC, constraints are not an afterthought.

They are part of the decision problem itself.

# 18. Control action

In [ ]:
plt.step(t_mpc, u_mpc, where="post")
plt.axhline(u_max, linestyle="--")
plt.axhline(-u_max, linestyle="--")

plt.xlabel("Time [s]")
plt.ylabel("Control input u [m³/s]")
plt.title("MPC control decisions")
plt.grid(True)
plt.show()

Because we used a finite control grid, the actions appear discrete.

This is actually useful conceptually for a human diver:

> choices can be thought of as small vent, no action, small inflation, and intermediate levels.

The mathematical MPC is a model of constrained decision logic, not necessarily a continuously automated valve controller.

# 19. Receding horizon visualized

Let's inspect one prediction horizon during the ascent.

At a selected time, show:

- current state;
- future reference;
- predicted trajectory under the chosen action.

In [ ]:
inspect_index = int(60.0 / dt_mpc)

state_inspect = states_mpc[inspect_index]
time_inspect = t_mpc[inspect_index]

u_inspect, _, pred_inspect = simple_mpc_action(
    state_inspect,
    time_inspect
)

future_t = time_inspect + np.arange(horizon_steps + 1) * dt_mpc
future_ref = np.array([reference(tt)[0] for tt in future_t])

plt.plot(future_t, future_ref, label="Future reference")
plt.plot(future_t, pred_inspect[:, 0], label="Predicted future")

plt.scatter(
    [time_inspect],
    [state_inspect[0]],
    s=60,
    label="Current state"
)

plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("One receding-horizon prediction")
plt.grid(True)
plt.legend()
plt.show()

The diver makes a decision now using a predicted future.

At the next step, this prediction is discarded and recomputed.

That continual replanning is the essence of MPC.

# 20. Prediction horizon

The horizon length matters.

A short horizon sees only the immediate future.

A long horizon can anticipate more distant consequences.

But longer prediction:

- requires more computation;
- depends more strongly on model accuracy.

This creates another tradeoff.

In [ ]:
for horizon in [4, 12, 30]:
    u_test, cost_test, _ = simple_mpc_action(
        state0,
        0.0,
        horizon=horizon
    )

    print(
        f"Horizon {horizon:2d} steps "
        f"({horizon*dt_mpc:.2f} s) "
        f"-> chosen u = {u_test:+.6f}"
    )

# 21. Human interpretation of prediction horizon

A diver also has an effective prediction horizon.

A novice may react mostly to:

> "What is happening now?"

A more experienced diver may anticipate:

> "If I keep this velocity for the next few seconds, what will happen to buoyancy and ascent rate?"

This is not literally an MPC optimization in the diver's brain.

But MPC is a useful mathematical analogy for **anticipatory control skill**.

# 22. Add an external disturbance

Suppose an unexpected upward disturbance occurs.

A predictive controller will observe the resulting new state and replan.

In [ ]:
def predict_one_step_disturbed(state, u, dt, disturbance_force=0.0):
    z, v, Vs = state

    u = np.clip(u, -u_max, u_max)

    a = acceleration(z, v, Vs, disturbance_force)

    v_next = v + a * dt
    z_next = max(z - v_next * dt, 0.0)
    Vs_next = np.clip(Vs + u * dt, Vs_min, Vs_max)

    return np.array([z_next, v_next, Vs_next])

In [ ]:
def simulate_mpc_with_disturbance(
    duration=220.0,
    dt=dt_mpc
):
    n = int(duration / dt) + 1
    t = np.linspace(0, duration, n)

    state_hist = np.zeros((n, 3))
    u_hist = np.zeros(n)

    state_hist[0] = np.array([z_start, 0.0, Vs_e])

    for k in range(n - 1):
        state = state_hist[k]

        u, _, _ = simple_mpc_action(
            state,
            t[k],
            dt=dt
        )

        disturbance = 0.0

        if 80.0 <= t[k] <= 84.0:
            disturbance = 20.0

        state_hist[k + 1] = predict_one_step_disturbed(
            state,
            u,
            dt,
            disturbance
        )

        u_hist[k] = u

    u_hist[-1] = u_hist[-2]

    return t, state_hist, u_hist

In [ ]:
t_d, states_d, u_d = simulate_mpc_with_disturbance()

z_d = states_d[:, 0]
z_ref_d = np.array([reference(tt)[0] for tt in t_d])

plt.plot(t_d, z_ref_d, label="Reference")
plt.plot(t_d, z_d, label="With disturbance")
plt.axvspan(80, 84, alpha=0.15)

plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("MPC replans after a disturbance")
plt.grid(True)
plt.legend()
plt.show()

The controller does not need to know the disturbance in advance.

It detects the changed state and replans from the new condition.

# 23. MPC vs fixed feedback

A conventional feedback controller may use:

$$
u=K_ze_z-K_ve_v.
$$

MPC instead asks:

> Which feasible action gives the best predicted future?

Feedback focuses primarily on the current error.

MPC explicitly includes future consequences and constraints.

# 24. Constraints can conflict

Suppose:

- the diver is far from the desired trajectory;
- the velocity is already near its allowed limit;
- the actuator is saturated.

The controller cannot satisfy every objective perfectly.

MPC resolves this by minimizing a weighted cost while respecting hard or penalized constraints.

This is an optimization problem.

# 25. Why the cost weights matter

The parameters:

$$
q_z,\quad q_v,\quad r_u
$$

express priorities.

Large $q_z$:

> track depth aggressively.

Large $q_v$:

> prioritize velocity tracking.

Large $r_u$:

> avoid large control actions.

Changing these weights changes the character of the decision-making.

In [ ]:
print("Current weights:")
print("q_z =", q_z)
print("q_v =", q_v)
print("r_u =", r_u)

# 26. MPC and human skill

The MPC analogy helps separate several aspects of skilled diving control.

### Estimation

What is my current depth and velocity?

### Prediction

If I continue like this, what happens next?

### Constraints

What motions and control actions are acceptable?

### Optimization

Which correction best balances competing goals?

### Replanning

What has changed since my last decision?

This is a rich control-theory interpretation of diver behavior.

# 27. Where the Kalman filter fits

In this notebook, MPC used the true state.

A realistic architecture would use:

$$
\hat x_k
$$

from the Kalman filter.

Then:

```text
sensor
  |
  v
Kalman filter
  |
  v
state estimate
  |
  v
MPC / predictive decision model
  |
  v
diver action
```

This combines estimation, prediction and constrained control.

# 28. Important practical interpretation

In DiveLab, MPC is best understood as a **normative model**:

> what would a rational predictive controller do under a simplified model and explicit constraints?

It does not imply that recreational diving should be automated.

It provides a framework for studying:

- anticipation;
- delayed consequences;
- competing objectives;
- safety constraints;
- bounded corrective actions.

# Exercises

### 1. Shorter prediction horizon

Try:

```python
horizon_steps = 4
```

How does the behavior change?

### 2. Longer horizon

Try:

```python
horizon_steps = 30
```

Does the controller become more anticipatory?

### 3. Stricter velocity constraint

Reduce:

```python
v_max
```

How does the chosen action change?

### 4. Increase control penalty

Increase:

```python
r_u
```

Does the controller become less aggressive?

### 5. Increase tracking priority

Increase:

```python
q_z
```

What happens to tracking and control effort?

# Challenge — optimize a full control sequence

Our simple MPC assumes one constant action over the prediction horizon.

A more complete MPC optimizes:

$$
u_0,u_1,\ldots,u_{N-1}
$$

simultaneously.

Implement a small brute-force or numerical optimization for a short horizon.

Then apply only:

$$
u_0
$$

and re-optimize at the next timestep.

This is true receding-horizon optimization.

In [ ]:
# Your code here

# Summary

In this notebook we introduced Model Predictive Control as a model of constrained, anticipatory diver decision-making.

We learned that MPC:

- predicts future trajectories;
- evaluates possible control actions;
- includes constraints explicitly;
- minimizes a cost function;
- applies only the first action;
- replans at every step.

We also clarified an important DiveLab interpretation:

> **the controller is the diver.**

The algorithms are mathematical models of sensing, estimation, anticipation and corrective action.

### Core MPC loop

$$
\boxed{
\text{Estimate}
\rightarrow
\text{Predict}
\rightarrow
\text{Choose}
\rightarrow
\text{Act}
\rightarrow
\text{Observe}
\rightarrow
\text{Repeat}
}
$$

### Next

Notebook 13 can introduce a **human-in-the-loop model** explicitly:

> How do reaction time, perception thresholds, imperfect decisions and intermittent control affect buoyancy stability?

That would make the diver-as-controller concept fully explicit.